# 01 – Auditoria dos dados

Inspeção, limpeza, agregação e validação de contagens (etapas 1 a 6 do fluxo). Sem os relatórios do demandante em `dados/bruto/`, o notebook roda em **modo demonstração**, com dados sintéticos, apenas para provar que o pipeline executa de ponta a ponta. Nenhum número do modo demonstração é resultado do projeto.

In [1]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config
from src.dados import carregar_bruto, limpar, agregar_mensal, validar_totais
from src.demo import gerar_registros_sinteticos, ROTULO_DEMO
pd.set_option("display.width", 140)

## 1. Obter e preservar
O notebook só lê `dados/bruto/`, nunca grava nele.

In [2]:
bruto = carregar_bruto()
MODO_DEMO = bruto is None
if MODO_DEMO:
    bruto = gerar_registros_sinteticos()
    print(ROTULO_DEMO)
else:
    print("Dados reais carregados de dados/bruto/")
print("Dimensões:", bruto.shape)
bruto.dtypes

MODO DEMONSTRAÇÃO: dados sintéticos, não são dados do LAPAE.
Dimensões: (14358, 4)


id_registro              int64
unidade                    str
municipio                  str
data_chegada    datetime64[us]
dtype: object

## 2. Inspecionar
Chaves, categorias, ausências e duplicidades.

In [3]:
print("Unidades (grafias brutas):", bruto[config.COL_UNIDADE].nunique())
print("Datas ausentes:", int(pd.to_datetime(bruto[config.COL_DATA], errors="coerce").isna().sum()))
if config.COL_ID in bruto.columns:
    print("Registros com id repetido:", int(bruto[config.COL_ID].duplicated().sum()))
bruto.head()

Unidades (grafias brutas): 16
Datas ausentes: 5
Registros com id repetido: 15


,id_registro,unidade,municipio,data_chegada
0,750,Unidade 02,Município 2,2025-10-04
1,6671,Unidade 06,Município 2,2026-01-16
2,10763,Unidade 08,Município 4,2026-07-24
3,10827,Unidade 08,Município 4,2026-08-23
4,1984,Unidade 02,Município 2,2026-06-29


## 3. Limpar, com contagens antes e depois

In [4]:
limpo, log = limpar(bruto)
log

,etapa,linhas_antes,linhas_depois,chaves_unicas_antes,chaves_unicas_depois
0,normalizar unidade e município,14358,14358,10,10
1,descartar datas inválidas ou ausentes,14358,14353,10,10
2,remover duplicatas por id_registro,14353,14338,14338,14338


## 4. Transformar: agregar unidade × mês (grade completa; ausência não vira zero)

In [5]:
mensal = agregar_mensal(limpo)
mensal.head(12)

,unidade,mes,exames,municipio
0,Unidade 01,2025-10-01,41,Município 1
1,Unidade 01,2025-11-01,61,Município 1
2,Unidade 01,2025-12-01,42,Município 1
3,Unidade 01,2026-01-01,57,Município 1
4,Unidade 01,2026-02-01,63,Município 1
5,Unidade 01,2026-03-01,62,Município 1
6,Unidade 01,2026-04-01,65,Município 1
7,Unidade 01,2026-05-01,62,Município 1
8,Unidade 01,2026-06-01,63,Município 1
9,Unidade 01,2026-07-01,57,Município 1


## 5. Validar totais

In [6]:
resultado = validar_totais(limpo, mensal)
print(resultado)
assert resultado["totais_conferem"], "Total de registros limpos difere da soma mensal"

{'registros_limpos': 14338, 'soma_base_mensal': 14338, 'totais_conferem': True, 'celulas_grade': 120, 'celulas_ausentes': 0, 'unidades': 10, 'meses': 12, 'unidades_sem_municipio': 0}


## 6. Salvar a base analítica
A base processada de demonstração recebe o sufixo `_DEMO` e não se mistura com a base real.

In [7]:
nome = "base_mensal_DEMO.csv" if MODO_DEMO else "base_mensal.csv"
mensal.to_csv(config.PROCESSADO / nome, index=False)
print("Gravado:", Path("dados/processado") / nome)

Gravado: dados/processado/base_mensal_DEMO.csv
